In [1]:
using LowLevelFEM, LinearAlgebra

In [2]:
openGeometry("box.geo")

In [3]:
#openPreProcessor()

In [4]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [11]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=true)

 11.331194 seconds (172.83 k allocations: 230.283 MiB, 1.73% gc time, 1.15% compilation time)


5

In [24]:
@time C = contact(u, master="master", slave="slave", leaf_size=1)

  0.191356 seconds (67.28 k allocations: 7.987 MiB)


Contact("slave" -> "master", 1149 candidate nodes, 603 active, stick=603, slip=0, G=(3447, 26790), C=(3447, 3447))

In [25]:
@time updateContact!(C, 1.3u)

  0.151125 seconds (67.27 k allocations: 8.183 MiB)


Contact("slave" -> "master", 1149 candidate nodes, 712 active, stick=712, slip=0, G=(3447, 26790), C=(3447, 3447))

In [26]:
@time updateContact!(C, 1.01u)

  0.125698 seconds (67.27 k allocations: 8.180 MiB)


Contact("slave" -> "master", 1149 candidate nodes, 604 active, stick=604, slip=0, G=(3447, 26790), C=(3447, 3447))

In [27]:
for ls in (1, 2, 4, 8)
    @time contact(
        u,
        master="master",
        slave="slave",
        leaf_size=ls
    )
end

  0.179146 seconds (67.27 k allocations: 7.986 MiB)
  0.188570 seconds (64.32 k allocations: 7.884 MiB)
  0.234406 seconds (62.72 k allocations: 7.822 MiB)
  0.283895 seconds (61.92 k allocations: 7.789 MiB, 1.72% gc time)


In [28]:
C.gap

elementwise ScalarField
[[0.11748664025768038; 0.10125044128166223; … ; 0.0969625160407128; 0.10520352967442782;;], [0.10129836059057812; 0.11748664025768038; … ; 0.10520352967442782; 0.09698953572322644;;], [0.10129954162619695; 0.11752283778270554; … ; 0.10521551140617136; 0.0969845352375229;;], [0.11752283778270554; 0.10124444273701104; … ; 0.09695981074649018; 0.10521551140617136;;], [0.10124496497632958; 0.11748531846133066; … ; 0.10520526864880721; 0.09695913083218732;;], [0.11748531846133066; 0.10130639094925327; … ; 0.0969945785018006; 0.10520526864880721;;], [0.1174280111743386; 0.10125972502619209; … ; 0.0969520853788312; 0.10517276738958868;;], [0.10128270706190501; 0.1174280111743386; … ; 0.10517276738958868; 0.09697797918354453;;], [0.0879659412935301; 0.10129954162619695; … ; 0.09009218477198191; 0.08265372593758594;;], [0.07806130360761084; 0.10129954162619695; … ; 0.0969845352375229; 0.08544278179048954;;]  …  [0.07807802546680523; 0.07806759625718933; … ; 0.08550096321

In [29]:
C.G[:,:]

3447×26790 SparseArrays.SparseMatrixCSC{Float64, Int64} with 40190 stored entries:
⎡⣿⡟⠀⠀⢸⣷⣶⠀⠀⠲⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠛⠃⠀⠀⢸⣿⣶⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠛⠂⠀⠀⠘⠛⠛⠀⠀⠀⠀⠀⠈⠓⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [30]:
showElementResults(C.n, name="n")

1

In [31]:
showElementResults(C.t1, name="t1")

2

In [32]:
showElementResults(C.t2, name="t2")

3

In [33]:
showElementResults(C.gap, name="gap", visible=false)

4

In [34]:
C.active

1149-element BitVector:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 1
 0
 1
 1
 1
 1
 0
 1
 0
 0
 0
 1

In [35]:
C.state

1149-element Vector{UInt8}:
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
    ⋮
 0x01
 0x00
 0x01
 0x01
 0x01
 0x01
 0x00
 0x01
 0x00
 0x00
 0x00
 0x01

In [36]:
openPostProcessor()